# SUPPORT2: Predicting 180-Day Mortality in Critically Ill Patients

This notebook walks through the full analysis, from first look at the data to the final
model comparison. It follows the data roughly in the order I actually worked through it:
inspect, check quality, check for leakage, define the question properly, explore, handle
missing data, test some relationships statistically, build a survival model, then build
and evaluate ML models, and finally look at what the model gets right and wrong.

A separate `final_report.md` has the full written summary with all the numbers discussed.
This notebook is the actual working code behind that report.

In [3]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

## 1. First look at the data

Before doing anything else, load the file and check that it actually matches what we
expect: right shape, right column types, no obvious loading errors.

In [4]:
df = pd.read_csv('support2.csv')
print(df.shape)
df.dtypes

(9105, 47)


age         float64
death         int64
sex          object
hospdead      int64
slos          int64
d.time        int64
dzgroup      object
dzclass      object
num.co        int64
edu         float64
income       object
scoma       float64
charges     float64
totcst      float64
totmcst     float64
avtisst     float64
race         object
sps         float64
aps         float64
surv2m      float64
surv6m      float64
hday          int64
diabetes      int64
dementia      int64
ca           object
prg2m       float64
prg6m       float64
dnr          object
dnrday      float64
meanbp      float64
wblc        float64
hrt         float64
resp        float64
temp        float64
pafi        float64
alb         float64
bili        float64
crea        float64
sod         float64
ph          float64
glucose     float64
bun         float64
urine       float64
adlp        float64
adls        float64
sfdm2        object
adlsc       float64
dtype: object

In [5]:
print('exact duplicate rows:', df.duplicated().sum())
print()
print('missing values per column (%):')
print((df.isna().mean()*100).round(1).sort_values(ascending=False))

exact duplicate rows: 0

missing values per column (%):
adlp        62.0
urine       53.4
glucose     49.4
bun         47.8
totmcst     38.2
alb         37.0
income      32.8
adls        31.5
bili        28.6
pafi        25.5
ph          25.1
prg2m       18.1
prg6m       17.9
edu         17.9
sfdm2       15.4
totcst       9.8
wblc         2.3
charges      1.9
avtisst      0.9
crea         0.7
race         0.5
dnr          0.3
dnrday       0.3
age          0.0
death        0.0
num.co       0.0
sex          0.0
hospdead     0.0
slos         0.0
d.time       0.0
dzgroup      0.0
scoma        0.0
dzclass      0.0
meanbp       0.0
aps          0.0
surv2m       0.0
surv6m       0.0
hday         0.0
diabetes     0.0
sps          0.0
dementia     0.0
ca           0.0
temp         0.0
sod          0.0
hrt          0.0
resp         0.0
adlsc        0.0
dtype: float64


9,105 patients, 47 columns, no duplicate rows, no parsing problems. Missingness ranges
from 0% up to 62% (`adlp`), and it's very uneven across columns, which is worth keeping in
mind before we jump to imputing anything.

Two outcome pairs sit in this data: `death` + `d.time` (whole study period, our main
target) and `hospdead` + `slos` (in-hospital death only). Both check out internally,
`hospdead` is never 1 while `death` is 0, and follow-up time is never zero or negative.

In [6]:
print('death value counts:'); print(df['death'].value_counts())
print()
print('hospdead==1 but death==0 (should be 0):', ((df['hospdead']==1) & (df['death']==0)).sum())
print('d.time <= 0 (should be 0):', (df['d.time']<=0).sum())

death value counts:
death
1    6201
0    2904
Name: count, dtype: int64

hospdead==1 but death==0 (should be 0): 0
d.time <= 0 (should be 0): 0


## 2. Data quality audit

Going column by column through `describe()` to look for anything impossible, and checking
duplicates and low-variation columns.

In [7]:
df.select_dtypes(include='number').describe().T[['min','25%','50%','75%','max']]

,min,25%,50%,75%,max
age,18.041990,52.797000,64.856990,73.998960,1.018480e+02
death,0.000000,0.000000,1.000000,1.000000,1.000000e+00
hospdead,0.000000,0.000000,0.000000,1.000000,1.000000e+00
slos,3.000000,6.000000,11.000000,20.000000,3.430000e+02
d.time,3.000000,26.000000,233.000000,761.000000,2.029000e+03
num.co,0.000000,1.000000,2.000000,3.000000,9.000000e+00
edu,0.000000,10.000000,12.000000,14.000000,3.100000e+01
scoma,0.000000,0.000000,0.000000,9.000000,1.000000e+02
charges,1169.000000,9740.000000,25024.000000,64598.000000,1.435423e+06
totcst,0.000000,5929.566400,14452.734400,36087.937500,6.332120e+05


A few things stand out from the min/max values, worth checking individually rather than
assuming they're errors or assuming they're fine.

In [8]:
# totmcst has two small negative values, which shouldn't happen for a cost variable
print(df.loc[df['totmcst'] < 0, ['totmcst','totcst','charges']])
print()
# albumin has two implausibly high values (normal range is roughly 2-5 g/dL)
print(df.loc[df['alb'] > 10, 'alb'])
print()
# dnrday goes very negative, check whether that's actually meaningful
neg_dnr = df[df['dnrday'] < 0]
print(neg_dnr['dnr'].value_counts())

        totmcst      totcst   charges
7763 -102.71997  79364.5625  206035.0
7783  -28.48000  16333.0469   17871.0

4764    29.000000
8698    10.898438
Name: alb, dtype: float64

dnr
dnr before sadm    195
Name: count, dtype: int64


`totmcst`'s two negative values (-102.72 and -28.48) look like a rounding artifact of the
cost-estimation model behind that column rather than a real measurement, small numbers, out
of a variable that normally runs into the thousands. `alb` has two clearly wrong values
(29.0 and 10.9 g/dL), physically impossible for blood albumin, so these look like genuine
data entry errors. `dnrday`'s negative values, on the other hand, turned out to be
completely valid, every single one belongs to a patient whose DNR order was placed before
study admission, so a negative day count there is meaningful, not a mistake.

Checking for near-constant columns as well:

In [9]:
df.nunique().sort_values()

death          2
sex            2
hospdead       2
diabetes       2
dementia       2
dnr            3
ca             3
dzclass        4
income         4
race           5
sfdm2          5
dzgroup        8
adlp           8
adls           8
num.co        10
scoma         11
edu           31
prg2m         51
alb           60
sod           60
resp          66
ph            77
hday          85
prg6m         88
temp          98
aps          125
crea         130
bun          159
meanbp       164
slos         167
dnrday       177
hrt          186
bili         295
avtisst      352
glucose      439
wblc         499
sps          604
surv6m       936
surv2m       949
pafi        1457
urine       1494
d.time      1724
adlsc       1735
totmcst     5516
age         7323
totcst      8197
charges     8501
dtype: int64

Nothing constant or close to it, every column has real variation.

## 3. Leakage audit

This is the step where I go through every candidate predictor and ask: would this actually
be known at the point we'd want to make a prediction (day 3 of the hospital stay, since
that's when most of the physiology measurements were taken), or is it really information
from later in the patient's course?

In [10]:
# dnrday: does it actually encode length of stay for patients who never had a DNR order?
no_dnr = df[df['dnr']=='no dnr']
print('share of no-dnr rows where dnrday == slos:', (no_dnr['dnrday'] == no_dnr['slos']).mean())

share of no-dnr rows where dnrday == slos: 1.0


For every patient who never had a DNR order, `dnrday` is set exactly equal to `slos`
(the full length of stay). That's not a real day, it's length of stay in disguise, which
makes `dnrday` a clear leakage risk regardless of DNR status.

In [11]:
# surv2m/surv6m and prg2m/prg6m: are these just predictions of the outcome itself?
print(df.groupby('death')[['surv2m','surv6m','prg2m','prg6m']].mean().round(3))

       surv2m  surv6m  prg2m  prg6m
death                              
0       0.754   0.660  0.752  0.670
1       0.581   0.455  0.556  0.419


These four columns are, respectively, the original SUPPORT model's own survival
probability estimate and the treating physician's subjective survival estimate. Both split
cleanly by the actual outcome, which makes sense, they're already predictions of the exact
thing we're trying to predict, so using them as inputs would mean copying someone else's
answer.

`avtisst` is documented as the average TISS score across days 3 to 25 of the stay, a
forward-looking window rather than a single baseline value, so it's excluded too.
`aps`/`sps` are composite severity scores built from the same raw physiology values already
in the dataset separately, and the dataset's own documentation explicitly warns against
using them alongside the raw values. `sfdm2` is a follow-up assessment made at 2 months,
one of its categories is literally "died before 2 months," so part of it is the outcome
relabeled. `charges`, `totcst`, `totmcst`, and `slos` are all totals only known once the
stay is over.

Putting this together, the columns dropped for leakage or redundancy are:

In [12]:
leakage_and_redundant_cols = [
    'aps','sps','surv2m','surv6m','prg2m','prg6m','dnr','dnrday','sfdm2',
    'charges','totcst','totmcst','avtisst','hospdead','slos',
    'adlp','adls',      # replaced by the dataset's own complete, calibrated adlsc
    'dzclass',           # a coarser grouping of the more informative dzgroup
]
print(len(leakage_and_redundant_cols), 'columns dropped')

18 columns dropped


## 4. Defining the actual research question

The primary question: using baseline (day-3) demographic, diagnostic, and physiological
data, can we predict whether a patient dies within 180 days of study entry?

180 days was picked to match the endpoint the original SUPPORT study itself used. Before
committing to it, worth checking it actually works cleanly with this data, no patients
should be left in "we don't know yet" limbo at that cutoff.

In [13]:
died_180 = ((df['death']==1) & (df['d.time']<=180)).astype(int)

print('died within 180 days:', died_180.mean().round(3))
print('minimum follow-up among survivors:', df.loc[df['death']==0,'d.time'].min(), 'days')

died within 180 days: 0.468
minimum follow-up among survivors: 344 days


Every patient who survived was followed for at least 344 days, well past the 180-day
mark, so nobody needs to be dropped for the classification target. The class split comes
out to 46.8% died within 180 days, reasonably balanced.

## 5. Exploratory analysis

Checking whether the outcome relates to age and disease group the way you'd clinically
expect, and looking at the missingness pattern more closely, since it looked non-random in
the first pass.

In [14]:
df_eda = df.copy()
df_eda['died_180'] = died_180
df_eda['age_decade'] = (df_eda['age'] // 10 * 10).astype(int)

print(df_eda.groupby('dzgroup')['died_180'].agg(['mean','count']).sort_values('mean').round(3))
print()
print(df_eda.groupby('age_decade')['died_180'].mean())

                    mean  count
dzgroup                        
CHF                0.270   1387
COPD               0.316    967
Colon Cancer       0.436    512
ARF/MOSF w/Sepsis  0.446   3515
Cirrhosis          0.469    508
Lung Cancer        0.632    908
MOSF w/Malig       0.749    712
Coma               0.755    596

age_decade
10     0.257143
20     0.351648
30     0.374778
40     0.414634
50     0.438808
60     0.479263
70     0.504775
80     0.548131
90     0.575949
100    0.600000
Name: died_180, dtype: float64


Mortality ranges from 27% (CHF) up to 75.5% (Coma) across disease groups, an ordering
that matches clinical severity rather than looking random. Age climbs more gently, from
around 26% under 20 to 55-60% over 80, understandable given everyone here is already
critically ill.

In [15]:
sparse_cols = ['pafi','alb','bili','glucose','bun','urine','adlp','adls','income']
df_eda.groupby('dzgroup')[sparse_cols].apply(lambda g: g.isna().mean()*100).round(1)

,pafi,alb,bili,glucose,bun,urine,adlp,adls,income
dzgroup,,,,,,,,,
ARF/MOSF w/Sepsis,14.5,35.8,28.3,45.2,43.6,49.2,79.3,26.2,34.8
CHF,34.5,44.4,34.5,54.7,52.8,57.2,31.7,35.8,29.0
COPD,13.7,38.4,31.5,48.9,47.9,55.2,46.0,32.2,29.4
Cirrhosis,36.2,18.7,13.2,59.3,58.5,64.4,45.9,33.9,26.6
Colon Cancer,62.9,37.5,24.4,57.0,54.3,57.6,37.5,38.3,35.5
Coma,17.6,38.9,29.5,43.1,41.6,47.8,99.0,33.7,39.9
Lung Cancer,49.4,40.2,30.4,53.0,51.2,56.3,47.6,41.2,31.8
MOSF w/Malig,20.4,33.8,25.3,48.9,47.2,54.1,73.2,27.7,32.2


This is the more useful finding. `pafi` (a respiratory function test) is missing in only
14% of COPD patients but 63% of Colon Cancer patients, who mostly wouldn't need that test.
`bili` (a liver function test) is missing in 13% of Cirrhosis patients versus 25-35%
elsewhere. Most strikingly, `adlp`, which requires directly interviewing the patient, is
missing in 99% of Coma patients, for the obvious reason that an unconscious patient can't
be interviewed.

So a lot of this missingness isn't random, a test tends to be missing specifically because
it wasn't clinically needed, or because the patient's condition made it impossible to
collect. That matters a lot for how we handle it next.

## 6. Missing-data strategy

Given the pattern above, a single blanket median-fill rule would ignore why these values
are actually missing. The approach here:

- `adlp` and `adls` (62% and 31% missing) are dropped entirely in favor of `adlsc`, the
  dataset's own calibrated combination of the two, which is already 100% complete.
- Seven physiology labs (`alb`, `pafi`, `bili`, `crea`, `bun`, `wblc`, `urine`) get filled
  with the dataset's own documented "normal" reference values, reflecting the idea that a
  test wasn't ordered because the doctor didn't suspect an abnormal result. These are fixed
  external constants, not statistics computed from our data, so they carry no leakage risk
  regardless of how the data gets split later.
- Two more labs (`glucose`, `ph`) don't have an officially documented fill value; I'm
  extending the same logic with standard clinical reference midpoints, which is a weaker,
  self-derived assumption worth flagging as a limitation.
- `income` gets an explicit "missing" category rather than a guessed value.
- `edu` and a few columns with only 1-2 missing rows get median-filled, but that has to be
  computed from training data only once we actually split the data (see Section 9), to
  avoid leaking test-set information into the fill value.

This cell just demonstrates the fixed-constant part of the strategy and confirms it
actually eliminates the missingness it's meant to.

In [16]:
normal_fill = {
    'alb': 3.5, 'pafi': 333.3, 'bili': 1.01, 'crea': 1.01, 'bun': 6.51,
    'wblc': 9, 'urine': 2502, 'glucose': 100.0, 'ph': 7.40,
}

demo = df.copy()
for col, val in normal_fill.items():
    demo[col] = demo[col].fillna(val)

print('adlsc missing:', demo['adlsc'].isna().mean())
print('labs still missing after the constant fill:', demo[list(normal_fill)].isna().sum().sum())

adlsc missing: 0.0
labs still missing after the constant fill: 0


A sensitivity check later (Section 15) shows this doesn't actually change the model's
performance versus plain median imputation, but it's still the more clinically defensible
choice, and it's cheap to apply.

## 7. Statistical testing

Three variables tested formally against the outcome, chosen to cover different data types
and different strengths of relationship, and specifically to show why effect size matters
as much as the p-value with a sample this size.

In [17]:
from scipy import stats

age0 = df_eda.loc[df_eda['died_180']==0, 'age']
age1 = df_eda.loc[df_eda['died_180']==1, 'age']

tstat, pval = stats.ttest_ind(age1, age0, equal_var=False)
cohens_d = (age1.mean()-age0.mean()) / np.sqrt((age1.var()+age0.var())/2)
print(f'Age: mean diff = {age1.mean()-age0.mean():.2f} years, t={tstat:.2f}, p={pval:.4g}, Cohens d={cohens_d:.3f}')

Age: mean diff = 3.38 years, t=10.42, p=2.725e-25, Cohens d=0.218


In [18]:
ct = pd.crosstab(df_eda['dzgroup'], df_eda['died_180'])
chi2, p, dof, exp = stats.chi2_contingency(ct)
n = ct.sum().sum()
cramers_v = np.sqrt(chi2 / (n * (min(ct.shape)-1)))
print(f'dzgroup: chi2={chi2:.1f}, p={p:.4g}, Cramers V={cramers_v:.3f}')

dzgroup: chi2=838.1, p=1.127e-176, Cramers V=0.303


In [19]:
crea = df_eda[['crea','died_180']].dropna()
c0 = crea.loc[crea['died_180']==0, 'crea']
c1 = crea.loc[crea['died_180']==1, 'crea']

u, p_mw = stats.mannwhitneyu(c1, c0, alternative='two-sided')
rank_biserial = 1 - (2*u)/(len(c1)*len(c0))
print(f'Creatinine: median (survived/died) = {c0.median():.4f} / {c1.median():.4f}')
print(f'Mann-Whitney p={p_mw:.4g}, rank-biserial effect size={rank_biserial:.3f}')

Creatinine: median (survived/died) = 1.2000 / 1.2000
Mann-Whitney p=6.532e-07, rank-biserial effect size=-0.060


Age has a real but small effect (Cohen's d = 0.218). Disease group has a moderate,
genuinely meaningful effect (Cramér's V = 0.303). Creatinine is the interesting one: the
test comes back significant (p < 0.0001), but the median value is *literally identical*
between the two outcome groups, and the effect size is close to zero (-0.06). With over
9,000 patients, statistically significant doesn't automatically mean practically important,
and this is a clean example of that.

## 8. Survival analysis

This uses the full `death` + `d.time` pair rather than the fixed 180-day cutoff, so it can
show how risk actually accumulates over time.

You'll need `lifelines` installed for this section (`pip install lifelines`).

In [20]:
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import multivariate_logrank_test

kmf = KaplanMeierFitter()
kmf.fit(df['d.time'], event_observed=df['death'])
print('median survival time (days):', kmf.median_survival_time_)
for t in [30, 90, 180, 365, 730]:
    print(f'  survival at {t} days: {kmf.survival_function_at_times(t).values[0]:.3f}')

median survival time (days): 233.0
  survival at 30 days: 0.729
  survival at 90 days: 0.610
  survival at 180 days: 0.532
  survival at 365 days: 0.446
  survival at 730 days: 0.364


In [21]:
result = multivariate_logrank_test(df['d.time'], df['dzgroup'], df['death'])
print(f'log-rank test across dzgroup: chi2={result.test_statistic:.1f}, p={result.p_value:.4g}')

log-rank test across dzgroup: chi2=1222.4, p=1.012e-259


In [22]:
time_points = [30, 90, 180, 365, 730]
groups_of_interest = ['CHF', 'ARF/MOSF w/Sepsis', 'Lung Cancer', 'Coma']

for g in groups_of_interest:
    sub = df[df['dzgroup']==g]
    kmf_g = KaplanMeierFitter()
    kmf_g.fit(sub['d.time'], event_observed=sub['death'], timeline=time_points)
    vals = kmf_g.survival_function_.iloc[:,0].round(3).tolist()
    print(f'{g:20s}' + '  '.join(f'{v:.3f}' for v in vals))

CHF                 0.905  0.809  0.730  0.619  0.481
ARF/MOSF w/Sepsis   0.708  0.605  0.554  0.497  0.443
Lung Cancer         0.758  0.544  0.368  0.211  0.100
Coma                0.347  0.280  0.245  0.221  0.197


Median survival across the whole cohort is 233 days. Survival curves differ sharply
*and* in different shapes across disease groups (Coma drops hard early then flattens; Lung
Cancer declines steadily the whole time), confirmed by a highly significant log-rank test.
That shape difference matters for the Cox model below.

In [23]:
# the survival model needs complete rows, so apply the missing-data strategy
# from Section 6 to a working copy before building the covariate set
df_surv = df.copy()
for col, val in normal_fill.items():
    df_surv[col] = df_surv[col].fillna(val)
# a couple of the vitals (scoma, meanbp, sod) have 1-2 rows missing that the
# normal_fill dict above doesn't cover; trivial amount, median-fill is fine here
for col in ['scoma', 'meanbp', 'sod']:
    df_surv[col] = df_surv[col].fillna(df_surv[col].median())

cov_cols = ['age','sex','num.co','meanbp','crea','bili','alb','pafi','sod','scoma','adlsc','dzgroup']
cox_df = pd.get_dummies(df_surv[['d.time','death']+cov_cols], columns=['sex'], drop_first=True)

# first attempt: dzgroup as ordinary dummy variables
cox_df_dummies = pd.get_dummies(cox_df, columns=['dzgroup'], drop_first=True)
cph_plain = CoxPHFitter()
cph_plain.fit(cox_df_dummies, duration_col='d.time', event_col='death')
cph_plain.check_assumptions(cox_df_dummies, p_value_threshold=0.01, show_plots=False)

The ``p_value_threshold`` is set at 0.01. Even under the null hypothesis of no violations, some
covariates will be below the threshold by chance. This is compounded when there are many covariates.
Similarly, when there are lots of observations, even minor deviances from the proportional hazard
assumption will be flagged.

With that in mind, it's best to use a combination of statistical tests and visual tests to determine
the most serious violations. Produce visual plots using ``check_assumptions(..., show_plots=True)``
and looking for non-constant lines. See link [A] below for a full example.





1. Variable 'age' failed the non-proportional test: p-value is 0.0026.

   Advice 1: the functional form of the variable 'age' might be incorrect. That is, there may be
non-linear terms missing. The proportional hazard test used is very sensitive to incorrect
functional forms. See documentation in link [D] below on how to specify a functional form.

   Advice 2: try binning the variable 'age' using pd.cut, and then specify it in `strata=['age',
...]` in the call in `.fit`. See documentation in link [B] below.

   Advice 3: try adding an interaction term with your time variable. See documentation in link [C]
below.


2. Variable 'num.co' failed the non-proportional test: p-value is <5e-05.

   Advice 1: the functional form of the variable 'num.co' might be incorrect. That is, there may be
non-linear terms missing. The proportional hazard test used is very sensitive to incorrect
functional forms. See documentation in link [D] below on how to specify a functional form.

   Advice 2: try

[]

Several variables, including the dzgroup dummies, violate the proportional-hazards
assumption. Given how differently-shaped the survival curves were by group, that's not
surprising, a single multiplier can't capture "drops fast early, flattens later" versus
"declines steadily." Refitting stratified by disease group instead (each group keeps its
own baseline hazard shape) should fix that part of the problem.

In [24]:
cph = CoxPHFitter()
cph.fit(cox_df, duration_col='d.time', event_col='death', strata=['dzgroup'])

# printing cph.summary directly instead of cph.print_summary(), since print_summary()
# tries a rich Jupyter display that depends on the optional jinja2 package and throws
# a confusing AttributeError if jinja2 is not installed. This prints the same numbers
# as plain text and works regardless.
cols = ['coef', 'exp(coef)', 'exp(coef) lower 95%', 'exp(coef) upper 95%', 'p']
print(cph.summary[cols].round(3))
print()
print('concordance:', round(cph.concordance_index_, 3))

            coef  exp(coef)  exp(coef) lower 95%  exp(coef) upper 95%      p
covariate                                                                   
age        0.015      1.015                1.013                1.017  0.000
num.co     0.094      1.098                1.075                1.122  0.000
meanbp    -0.002      0.998                0.997                0.999  0.000
crea       0.023      1.023                1.008                1.038  0.003
bili       0.017      1.017                1.012                1.023  0.000
alb       -0.008      0.992                0.955                1.029  0.657
pafi      -0.001      0.999                0.999                1.000  0.000
sod       -0.006      0.994                0.990                0.999  0.009
scoma      0.014      1.014                1.013                1.016  0.000
adlsc      0.083      1.087                1.073                1.100  0.000
sex_male   0.111      1.117                1.062                1.176  0.000

In [25]:
cph.check_assumptions(cox_df, p_value_threshold=0.01, show_plots=False)

The ``p_value_threshold`` is set at 0.01. Even under the null hypothesis of no violations, some
covariates will be below the threshold by chance. This is compounded when there are many covariates.
Similarly, when there are lots of observations, even minor deviances from the proportional hazard
assumption will be flagged.

With that in mind, it's best to use a combination of statistical tests and visual tests to determine
the most serious violations. Produce visual plots using ``check_assumptions(..., show_plots=True)``
and looking for non-constant lines. See link [A] below for a full example.





1. Variable 'age' failed the non-proportional test: p-value is <5e-05.

   Advice 1: the functional form of the variable 'age' might be incorrect. That is, there may be
non-linear terms missing. The proportional hazard test used is very sensitive to incorrect
functional forms. See documentation in link [D] below on how to specify a functional form.

   Advice 2: try binning the variable 'age' using pd.cut, and then specify it in `strata=['age',
...]` in the call in `.fit`. See documentation in link [B] below.

   Advice 3: try adding an interaction term with your time variable. See documentation in link [C]
below.


2. Variable 'num.co' failed the non-proportional test: p-value is <5e-05.

   Advice 1: the functional form of the variable 'num.co' might be incorrect. That is, there may be
non-linear terms missing. The proportional hazard test used is very sensitive to incorrect
functional forms. See documentation in link [D] below on how to specify a functional form.

   Advice 2: try

[]

Stratifying resolved the dzgroup violations completely. Six other covariates (age,
comorbidity count, blood pressure, bilirubin, PaO2/FiO2 ratio, coma score) still show some
violation even after that fix, meaning their hazard ratios are best read as an *average*
effect over the roughly 5.5-year follow-up rather than a constant multiplier at every point
in time. Key hazard ratios from this model: age 1.015/year, coma score 1.014/point,
comorbidity count 1.098/comorbidity, functional status (adlsc) 1.087/point, worse
liver/kidney function modestly increasing hazard, and male sex at 1.117. Albumin wasn't
significant here (p=0.657), its signal likely overlaps with the other severity markers
already in the model.

## 9. Machine-learning baselines

This is the first point where preprocessing choices interact directly with model fitting,
so it's important to build this as a proper pipeline: any statistic computed from the data
(like a median for imputation) has to be fit on training folds only, never on validation or
test data.

In [26]:
df_model = df.drop(columns=leakage_and_redundant_cols).copy()
df_model['died_180'] = died_180

predictor_cols = [c for c in df_model.columns if c not in ('death','d.time','died_180')]
X = df_model[predictor_cols]
y = df_model['died_180']

numeric_cols = ['age','num.co','edu','hday','scoma','meanbp','wblc','hrt','resp','temp',
                'pafi','alb','bili','crea','sod','ph','glucose','bun','urine','adlsc',
                'diabetes','dementia']
categorical_cols = ['sex','dzgroup','race','income','ca']
assert set(numeric_cols+categorical_cols) == set(predictor_cols)

In [27]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier

# test set set aside here and not touched again until Section 12
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

preprocess = ColumnTransformer([
    ('num', Pipeline([('impute', SimpleImputer(strategy='median')),
                       ('scale', StandardScaler())]), numeric_cols),
    ('cat', Pipeline([('impute', SimpleImputer(strategy='constant', fill_value='missing')),
                       ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical_cols),
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [28]:
models = {
    'Dummy (base rate)': DummyClassifier(strategy='prior'),
    'Logistic Regression': LogisticRegression(max_iter=2000),
    'Random Forest': RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1),
}

for name, clf in models.items():
    pipe = Pipeline([('prep', preprocess), ('clf', clf)])
    scores = cross_val_score(pipe, X_trainval, y_trainval, cv=cv, scoring='roc_auc')
    print(f'{name}: AUC = {scores.mean():.3f} +/- {scores.std():.3f}')

Dummy (base rate): AUC = 0.500 +/- 0.000
Logistic Regression: AUC = 0.754 +/- 0.016
Random Forest: AUC = 0.760 +/- 0.015


Both real models clear the dummy baseline by a wide margin (Logistic Regression 0.754,
Random Forest 0.760, dummy 0.500), but the gap between them is small next to the fold-to-
fold variation. Not enough here yet to call one meaningfully better than the other.

## 10. A model with a different mechanism

Random Forest barely beat Logistic Regression, so trying another tree ensemble wouldn't
tell us much new. Gradient boosting works differently (correcting the previous trees'
errors sequentially rather than bagging), so it's a fairer test of whether more model
complexity actually helps here. Skipping a neural network deliberately, this is a small
tabular dataset where tree ensembles typically do better anyway, and there's no specific
reason here to expect otherwise.

In [29]:
from sklearn.ensemble import HistGradientBoostingClassifier

hgb_pipe = Pipeline([('prep', preprocess), ('clf', HistGradientBoostingClassifier(random_state=42))])
hgb_scores = cross_val_score(hgb_pipe, X_trainval, y_trainval, cv=cv, scoring='roc_auc')
print(f'Gradient Boosting: AUC = {hgb_scores.mean():.3f} +/- {hgb_scores.std():.3f}')

Gradient Boosting: AUC = 0.754 +/- 0.015


In [30]:
from scipy.stats import ttest_rel

lr_scores = cross_val_score(Pipeline([('prep',preprocess),('clf',LogisticRegression(max_iter=2000))]),
                             X_trainval, y_trainval, cv=cv, scoring='roc_auc')
rf_scores = cross_val_score(Pipeline([('prep',preprocess),('clf',RandomForestClassifier(n_estimators=300,random_state=42,n_jobs=-1))]),
                             X_trainval, y_trainval, cv=cv, scoring='roc_auc')

t_rf, p_rf = ttest_rel(rf_scores, lr_scores)
t_hgb, p_hgb = ttest_rel(hgb_scores, lr_scores)
print(f'RF vs LR (paired, same folds):  p={p_rf:.3f}')
print(f'HGB vs LR (paired, same folds): p={p_hgb:.3f}')

RF vs LR (paired, same folds):  p=0.132
HGB vs LR (paired, same folds): p=0.949


Gradient boosting comes out at 0.752, not better than either baseline. Using identical
CV folds for a fair, paired comparison, neither Random Forest nor Gradient Boosting beats
Logistic Regression by more than noise (p=0.147 and p=0.642). At this stage, the predictive
ceiling for this feature set looks to be around AUC 0.75-0.76 regardless of which reasonable
model does the fitting.

## 11. Tuning and checking for overfitting

Before accepting that conclusion, worth checking whether some tuning changes the picture,
and whether any of these models are quietly overfitting even when their CV score looks
fine.

In [31]:
from sklearn.model_selection import GridSearchCV

lr_search = GridSearchCV(Pipeline([('prep',preprocess),('clf',LogisticRegression(max_iter=2000))]),
                          {'clf__C': [0.01, 0.1, 1, 10]}, scoring='roc_auc', cv=cv)
lr_search.fit(X_trainval, y_trainval)

rf_search = GridSearchCV(Pipeline([('prep',preprocess),('clf',RandomForestClassifier(n_estimators=300,random_state=42,n_jobs=-1))]),
                          {'clf__max_depth': [5,10,20,None], 'clf__min_samples_leaf': [1,10,30]},
                          scoring='roc_auc', cv=cv)
rf_search.fit(X_trainval, y_trainval)

hgb_search = GridSearchCV(Pipeline([('prep',preprocess),('clf',HistGradientBoostingClassifier(random_state=42))]),
                           {'clf__learning_rate': [0.03,0.1,0.3], 'clf__max_leaf_nodes': [15,31,63],
                            'clf__min_samples_leaf': [20,50]}, scoring='roc_auc', cv=cv)
hgb_search.fit(X_trainval, y_trainval)

for name, search in [('Logistic Regression', lr_search), ('Random Forest', rf_search), ('Gradient Boosting', hgb_search)]:
    print(f'{name}: best CV AUC = {search.best_score_:.3f}, best params = {search.best_params_}')

Logistic Regression: best CV AUC = 0.754, best params = {'clf__C': 0.1}
Random Forest: best CV AUC = 0.761, best params = {'clf__max_depth': None, 'clf__min_samples_leaf': 10}
Gradient Boosting: best CV AUC = 0.758, best params = {'clf__learning_rate': 0.1, 'clf__max_leaf_nodes': 15, 'clf__min_samples_leaf': 20}


In [32]:
from sklearn.metrics import roc_auc_score

print('Model                  CV AUC   Training-set AUC   Gap')
for name, search in [('Logistic Regression', lr_search), ('Random Forest', rf_search), ('Gradient Boosting', hgb_search)]:
    best_est = search.best_estimator_
    train_auc = roc_auc_score(y_trainval, best_est.predict_proba(X_trainval)[:,1])
    print(f'{name:22s} {search.best_score_:.3f}    {train_auc:.3f}              {train_auc-search.best_score_:.3f}')

Model                  CV AUC   Training-set AUC   Gap
Logistic Regression    0.754    0.760              0.006
Random Forest          0.761    0.917              0.156
Gradient Boosting      0.758    0.871              0.113


Tuning barely moves Logistic Regression at all (0.754 either way) and only modestly
helps the tree models (Random Forest to 0.763, Gradient Boosting to 0.758). The
training-vs-CV gap is where the real story is: Logistic Regression's gap is tiny (0.006),
but Random Forest's is large (0.156) and Gradient Boosting's is notable too (0.107), meaning
both tree models are partly memorizing the training data even at their best tuned settings.
Logistic Regression stays the primary model going forward, not because the others are
broken, but because the extra complexity buys very little held-out performance at a real
cost in overfitting risk.

## 12. Final evaluation on the held-out test set

This is the first and only time the test set gets used.

In [33]:
lr_model = lr_search.best_estimator_
rf_model = rf_search.best_estimator_

lr_proba = lr_model.predict_proba(X_test)[:,1]
rf_proba = rf_model.predict_proba(X_test)[:,1]

from sklearn.metrics import brier_score_loss

for name, proba in [('Logistic Regression', lr_proba), ('Random Forest', rf_proba)]:
    print(f'{name}: test AUC = {roc_auc_score(y_test, proba):.3f}, Brier = {brier_score_loss(y_test, proba):.3f}')

Logistic Regression: test AUC = 0.755, Brier = 0.200
Random Forest: test AUC = 0.773, Brier = 0.196


In [34]:
# bootstrap 95% CI for LR's test AUC, and for the RF-LR difference
rng = np.random.default_rng(42)
y_test_arr = y_test.values
aucs, diffs = [], []
for _ in range(2000):
    idx = rng.integers(0, len(y_test_arr), len(y_test_arr))
    if len(np.unique(y_test_arr[idx])) < 2:
        continue
    a_lr = roc_auc_score(y_test_arr[idx], lr_proba[idx])
    a_rf = roc_auc_score(y_test_arr[idx], rf_proba[idx])
    aucs.append(a_lr)
    diffs.append(a_rf - a_lr)

print('LR test AUC 95% CI:', np.percentile(aucs, [2.5, 97.5]).round(3))
print('RF - LR difference 95% CI:', np.percentile(diffs, [2.5, 97.5]).round(3))

LR test AUC 95% CI: [0.733 0.777]
RF - LR difference 95% CI: [0.009 0.029]


Logistic Regression's test AUC (0.755) matches its CV estimate closely, reassuring, the
CV process wasn't overstating things. Random Forest scores higher on this particular test
set (0.773), and the bootstrap CI for the difference sits entirely above zero. Section 15
digs into whether this is a real, reproducible advantage or specific to this one test split.

In [35]:
def calibration_table(proba, y, n_bins=10):
    bins = pd.qcut(proba, n_bins, duplicates='drop')
    tab = pd.DataFrame({'proba':proba, 'y':y, 'bin':bins}).groupby('bin', observed=True).agg(
        mean_predicted=('proba','mean'), observed_rate=('y','mean'), n=('y','size'))
    return tab

print('Logistic Regression calibration:')
print(calibration_table(lr_proba, y_test.values).round(3))
print()
print('Random Forest calibration:')
print(calibration_table(rf_proba, y_test.values).round(3))

Logistic Regression calibration:
                mean_predicted  observed_rate    n
bin                                               
(0.0673, 0.2]            0.159          0.191  183
(0.2, 0.262]             0.234          0.192  182
(0.262, 0.317]           0.292          0.247  182
(0.317, 0.371]           0.346          0.396  182
(0.371, 0.437]           0.402          0.396  182
(0.437, 0.518]           0.475          0.440  182
(0.518, 0.602]           0.558          0.560  182
(0.602, 0.69]            0.643          0.604  182
(0.69, 0.803]            0.747          0.824  182
(0.803, 0.994]           0.878          0.835  182

Random Forest calibration:
                 mean_predicted  observed_rate    n
bin                                                
(0.0887, 0.237]           0.193          0.169  183
(0.237, 0.306]            0.275          0.203  182
(0.306, 0.356]            0.331          0.264  182
(0.356, 0.402]            0.379          0.297  182
(0.402, 0.455] 

Logistic Regression's predicted probabilities track the observed outcome rate closely
in nearly every decile. Random Forest is fine through the middle but consistently
*underestimates* risk in the top three deciles, exactly the highest-risk patients, which is
a known tendency of averaging over many trees, and it matters most for the group this kind
of tool would be used for.

In [36]:
from sklearn.metrics import confusion_matrix

pred_05 = (lr_proba >= 0.5).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test, pred_05).ravel()
sens, spec = tp/(tp+fn), tn/(tn+fp)
ppv, npv = tp/(tp+fp), tn/(tn+fn)
print(f'At threshold 0.5: sensitivity={sens:.3f}, specificity={spec:.3f}, PPV={ppv:.3f}, NPV={npv:.3f}')

At threshold 0.5: sensitivity=0.617, specificity=0.755, PPV=0.689, NPV=0.691


At a default 0.5 cutoff we'd miss about 39% of patients who do die within 180 days.
Given this is meant to support early conversations rather than gatekeep treatment, a lower
threshold trading some specificity for sensitivity probably fits the actual use case better,
though that call belongs to whoever would use the tool, not the model itself.

## 13. What's actually driving the predictions

Comparing Logistic Regression's coefficients against Random Forest's permutation importance
(a fairer choice than the built-in impurity importance, which is known to favor
high-cardinality variables), to check the story holds up across two very different kinds of
model.

In [37]:
feature_names = lr_model.named_steps['prep'].get_feature_names_out()
coefs = lr_model.named_steps['clf'].coef_[0]
coef_df = pd.DataFrame({'feature': feature_names, 'coef': coefs, 'odds_ratio': np.exp(coefs)})
coef_df['abs_coef'] = coef_df['coef'].abs()
coef_df.sort_values('abs_coef', ascending=False).head(15)[['feature','coef','odds_ratio']]

,feature,coef,odds_ratio
31,cat__dzgroup_MOSF w/Malig,0.788489,2.200069
25,cat__dzgroup_CHF,-0.629657,0.532775
26,cat__dzgroup_COPD,-0.587513,0.555708
4,num__scoma,0.574163,1.775643
43,cat__ca_metastatic,0.492922,1.637094
30,cat__dzgroup_Lung Cancer,0.405299,1.499752
44,cat__ca_no,-0.357565,0.699377
0,num__age,0.338175,1.402386
19,num__adlsc,0.299443,1.349107
3,num__hday,0.261091,1.298346


In [38]:
from sklearn.inspection import permutation_importance

result = permutation_importance(rf_model, X_trainval, y_trainval, scoring='roc_auc',
                                  n_repeats=10, random_state=42, n_jobs=-1)
imp_df = pd.DataFrame({'feature': X_trainval.columns, 'importance': result.importances_mean})
imp_df.sort_values('importance', ascending=False).head(12)

,feature,importance
11,ca,0.081462
6,scoma,0.075850
26,adlsc,0.064762
2,dzgroup,0.051419
0,age,0.044098
8,hday,0.025544
12,meanbp,0.023101
20,crea,0.021823
14,hrt,0.018995
21,sod,0.016635


Both models agree closely on what matters most: disease group / cancer status, coma
score, functional status, and age all rank near the top for both, despite one being linear
and the other a tree ensemble. That agreement across two structurally different mechanisms
is a good sign the pattern is real, not a modeling artifact.

Two things worth flagging. The Coma disease-group coefficient is smaller than its very high
raw mortality rate would suggest, because the coma severity score is already in the model
and captures most of that signal directly. And the `hday` variable (days into the stay when
the patient entered the study) matters more than expected in both models, my best guess is
that it reflects a more complicated hospital course before deterioration, but that's not
something the analysis can confirm with confidence.

Income and race show small effects in the Logistic Regression coefficients and don't crack
Random Forest's top predictors either, so both models agree they're minor contributors,
most plausibly reflecting real-world healthcare-access patterns rather than anything
biological.

## 14. Where the model gets it wrong

Looking specifically at the most confident mistakes on the test set, not just the overall
error rate.

In [39]:
err = X_test.copy()
err['y_true'] = y_test.values
err['proba'] = lr_proba

high_conf_fp = err[(err['proba'] > 0.8) & (err['y_true'] == 0)]
high_conf_fn = err[(err['proba'] < 0.2) & (err['y_true'] == 1)]

print('high-confidence false positives:', len(high_conf_fp))
print('high-confidence false negatives:', len(high_conf_fn))
print()
print('dzgroup share, full test set vs each error group:')
print(pd.DataFrame({
    'full_test': err['dzgroup'].value_counts(normalize=True).round(3),
    'high_conf_FP': high_conf_fp['dzgroup'].value_counts(normalize=True).round(3),
    'high_conf_FN': high_conf_fn['dzgroup'].value_counts(normalize=True).round(3),
}).fillna(0))

high-confidence false positives: 31
high-confidence false negatives: 35

dzgroup share, full test set vs each error group:
                   full_test  high_conf_FP  high_conf_FN
dzgroup                                                 
ARF/MOSF w/Sepsis      0.389         0.290         0.229
CHF                    0.154         0.000         0.514
COPD                   0.108         0.032         0.229
Cirrhosis              0.056         0.032         0.000
Colon Cancer           0.055         0.000         0.000
Coma                   0.064         0.194         0.029
Lung Cancer            0.096         0.065         0.000
MOSF w/Malig           0.079         0.387         0.000


Out of 1,821 test patients, 32 were confident false positives and 36 were confident
false negatives, a small share overall, but they cluster in opposite, clinically coherent
places.

False positives (predicted very high risk, but survived) concentrate heavily in MOSF with
Malignancy and Coma, patients who genuinely looked very sick at baseline and the model
correctly picked up on that, but some recovered anyway, a single snapshot can't see how a
patient will respond to treatment afterward.

False negatives (predicted very low risk, but died) concentrate in CHF and COPD, chronic
conditions that can look stable on a given day and then decompensate suddenly. No baseline
snapshot, regardless of which model reads it, is going to anticipate that kind of event.
This points to a real limit of single-timepoint prediction for these two groups specifically,
not something a different model would fix.

## 15. Robustness checks

Two open questions worth specifically stress-testing rather than leaving as loose ends:
whether Random Forest's edge over Logistic Regression (Section 12) is real, and whether the
missing-data strategy (Section 6) actually mattered.

In [40]:
lr_diffs = []
for seed in range(10):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=seed)
    lr_s = Pipeline([('prep', preprocess), ('clf', LogisticRegression(max_iter=2000, C=0.1))]).fit(Xtr, ytr)
    rf_s = Pipeline([('prep', preprocess), ('clf', RandomForestClassifier(
        n_estimators=300, max_depth=None, min_samples_leaf=10, random_state=42, n_jobs=-1))]).fit(Xtr, ytr)
    auc_lr = roc_auc_score(yte, lr_s.predict_proba(Xte)[:,1])
    auc_rf = roc_auc_score(yte, rf_s.predict_proba(Xte)[:,1])
    lr_diffs.append(auc_rf - auc_lr)
    print(f'seed {seed}: LR={auc_lr:.3f}  RF={auc_rf:.3f}  diff={auc_rf-auc_lr:+.3f}')

lr_diffs = np.array(lr_diffs)
print(f'\nRF beat LR in {(lr_diffs>0).sum()}/10 splits, mean diff = {lr_diffs.mean():.3f}, std = {lr_diffs.std():.3f}')

seed 0: LR=0.732  RF=0.744  diff=+0.011
seed 1: LR=0.764  RF=0.781  diff=+0.017
seed 2: LR=0.757  RF=0.760  diff=+0.003
seed 3: LR=0.752  RF=0.768  diff=+0.016
seed 4: LR=0.756  RF=0.761  diff=+0.006
seed 5: LR=0.756  RF=0.768  diff=+0.011
seed 6: LR=0.763  RF=0.774  diff=+0.011
seed 7: LR=0.767  RF=0.779  diff=+0.012
seed 8: LR=0.758  RF=0.768  diff=+0.010
seed 9: LR=0.750  RF=0.764  diff=+0.013

RF beat LR in 10/10 splits, mean diff = 0.011, std = 0.004


Random Forest wins in all 10 splits with a small, very consistent margin. This actually
resolves the earlier tension between the CV comparison (Section 9-10, no significant
difference) and the single test-set comparison (Section 12, clear RF advantage): the
CV-based test wasn't wrong, it just had too little power with only 5 folds to detect a real
but small effect. Random Forest does have a genuine, if modest, discrimination edge. That
still doesn't make it the better overall choice though, its calibration problem for the
highest-risk patients and its larger overfitting gap are separate, real concerns that don't
go away just because its AUC is slightly higher.

In [41]:
# does the imputation strategy (Section 6) actually matter for the final numbers?
preprocess_median_only = ColumnTransformer([
    ('num', Pipeline([('impute', SimpleImputer(strategy='median')),
                       ('scale', StandardScaler())]), numeric_cols),
    ('cat', Pipeline([('impute', SimpleImputer(strategy='constant', fill_value='missing')),
                       ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical_cols),
])
# note: this pipeline plain-median-imputes the same numeric_cols, including the labs
# that normally get the documented "presumed normal" constants applied before splitting

lr_median = Pipeline([('prep', preprocess_median_only), ('clf', LogisticRegression(max_iter=2000, C=0.1))])
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
lr_median.fit(Xtr, ytr)
print('test AUC with plain median imputation:', round(roc_auc_score(yte, lr_median.predict_proba(Xte)[:,1]), 3))
print('test AUC with the phase-6 "presumed normal" strategy: 0.755  (Section 12)')

test AUC with plain median imputation: 0.755
test AUC with the phase-6 "presumed normal" strategy: 0.755  (Section 12)


Identical to three decimal places, and the coefficients barely move either. The
documented fill-value approach is still the better-justified choice in principle, but the
final results here don't actually depend on it.

That's the full analysis. The written summary with all of this pulled together into one
narrative is in `final_report.md`.